# Fig 2d–e: cell-type co-occurrence modules and their survival associations

Produces (`figures/paper/fig2de_celltype_modules/`, and the panels folder):
- **Fig 2d**: Pearson correlation of level-4 cell-type proportions across samples, Ward
  clustering, 25 modules cut at distance 1.0 (`fig2d_celltype_correlation_level4`, module
  membership `sc_celltype_level4_clusters.csv`; panels folder `Fig2d`, `Fig2dmodules`).
- **Fig 2e**: RMST difference at τ = 1.5 years, high vs low module score, adjusted for MGMT,
  age, sex and p53, for the 7 modules significant in the MGMT + age grid search
  (`fig2e_forest_rmst_modules`). Text: module 9 +2.0 mo, 4 −1.9 mo, 16 −1.2 mo, 7 +1.4 mo,
  24 +1.3 mo; † = also significant at τ = 3 years (text: 2, 4, 8).

Origin: `pipelines/NicheNet_v3.ipynb` (Spencer Watson), cells 162 (Fig 2d), 172 (grid search
adjusted for MGMT + age: picks threshold 1.0, τ = 1.5 y and, per module, the scoring and the
split) and 174 (Fig 2e forest), numbered as in the EFS copy `Final_Notebooks/NicheNet_v3.ipynb`
(163, 173, 175 in `pipelines/NicheNet_v3.ipynb`). Cells 164–170 and 173 (exploratory) are not
included.

How the analysis runs:
- The snRNA-seq cohort is `AllDataCenters.load_singlecell()` **without exclusions** (obs only):
  298 samples (IDH-mutant, relapse and excluded patients included; the Methods say 266). The
  modules are defined on the 298.
- Survival join: samples whose name ends in "a", minus that letter, matched to `Subject Id`
  of every row of the three processed clinical tables (one row per subject, baseline first).
  273 patients are fitted, some outside the 260-patient survival cohort.
- Covariates: `(x == level)`, so unknown MGMT, sex or p53 count as unmethylated, male,
  negative; age is the only covariate whose missing value drops a patient. The number of
  fitted patients this affects is printed.

Comparison: Fig 2e reproduces the saved output of cell 174 (all 7 modules, checked below); the
† at τ = 3 years, the fitted-patient counts and the corrected versions (survival cohort with
complete-case covariates; modules on the 266 paper samples) are additions for the response letter.

In [ ]:
import warnings

import anndata as ad
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.cluster.hierarchy import fcluster, linkage
from scipy.spatial.distance import squareform
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

from gbmcollaboration import AllDataCenters
from gbmcollaboration.paper import annotation, cohort, output
from gbmcollaboration.paper import config  # noqa: F401  (sets R_HOME for rpy2)

import rpy2.robjects as ro
from rpy2.robjects import pandas2ri
from rpy2.robjects.conversion import localconverter

warnings.filterwarnings("ignore")

In [ ]:
# Parameters
OUT = output.setup("fig2de_celltype_modules")

# Fig 2d (user config)
CT_KEY_L3 = "celltype_level3_curated"
CT_KEY_L4 = "celltype_level4_curated"
SAMPLE_KEY = "orig.ident"

EXCLUDE_PATTERNS = ["Doublet", "doublet"]

FIG_SIZE_L3 = 8
FIG_SIZE_L4 = 12
LABEL_FONTSIZE_L3 = 7
LABEL_FONTSIZE_L4 = 6
TITLE_FONTSIZE = 12

CMAP = "seismic"
VMIN = -1
VMAX = 1

CLUSTER_METHOD = "ward" # or "average", "complete"
CLUSTER_THRESHOLD_L3 = 1
CLUSTER_THRESHOLD_L4 = 1

SHOW_CBAR = False

# Fig 2e grid search (user config)
CLUSTER_THRESHOLDS = [0.7, 0.8, 0.9, 1.0, 1.2, 1.3, 1.5]

SURVIVAL_TYPE = "OS"

SCORING_METHODS = ["sum", "mean", "max", "pca"]
STRATIFICATIONS = ["median", "tertile", "quartile_vs_rest", "quartile_vs_quartile"]

TAU_OPTIONS = [1.5, 2.0, 2.5, 3.0]

P_THRESHOLD = 0.05

SAMPLE_ID_COL = "Subject Id"
CHRONOLOGY_COL = "Sample collection chronology"
TIME_COL_OS = "os_years"
EVENT_COL_OS = "os_event"
TIME_COL_PFS = "pfs_years"
EVENT_COL_PFS = "pfs_event"
MGMT_COL = "MGMT promoter methylation"
AGE_COL = "age_at_diagnosis"

# Fig 2e forest (config)
ADJUST_MODE   = "all"   # "mgmt", "mgmt_age", "all", or "none"
SEL_TAU       = 1.5
DAGGER_TAU    = 3.0     # the dagger of Fig 2e

TIME_COL       = "os_years"
EVENT_COL      = "os_event"
SEX_COL        = "Administrative gender"
P53_COL        = "p53"

# Fig 2e forest (aesthetics)
FP_FIG_W     = 12
FP_FIG_H     = 4
FP_DPI       = 300
FP_FS_TITLE  = 9
FP_FS_AXIS   = 8
FP_FS_HDR    = 8
FP_FS_ROW    = 8
FP_MARKER    = "D"
FP_MARKER_SZ = 5
FP_CAP_SZ    = 3
FP_LW        = 1.3
FP_ELW       = 1.1
FP_COL_POS   = "#2166AC"
FP_COL_NEG   = "#D6604D"
FP_COL_NS    = "#888888"
FP_SIG_ALPHA = 0.05
X_LABEL      = -0.02
X_EST        =  1.02
X_P          =  1.28

# Published Fig 2e: module -> (diff months, adjusted p)
PUBLISHED_FIG2E = {9: (2.02, 0.0008), 8: (1.94, 0.0022), 4: (-1.86, 0.0099), 2: (-1.80, 0.0129),
                   7: (1.44, 0.0301), 24: (1.29, 0.0367), 16: (-1.23, 0.0424)}

In [ ]:
# Functions

def is_excluded(ct_name):
    for pattern in EXCLUDE_PATTERNS:
        if pattern in str(ct_name):
            return True
    return False

def build_proportion_matrix(adata, ct_key, sample_key):
    obs = adata.obs[[sample_key, ct_key]].copy()
    obs[sample_key] = obs[sample_key].astype(str)
    obs[ct_key] = obs[ct_key].astype(str)
    
    counts = obs.groupby([sample_key, ct_key]).size().unstack(fill_value=0)
    keep_cols = [c for c in counts.columns if not is_excluded(c)]
    counts = counts[keep_cols]
    props = counts.div(counts.sum(axis=1), axis=0)
    
    return props

def clean_corr(corr):
    corr = corr.replace([np.inf, -np.inf], np.nan)
    corr = corr.dropna(axis=0, how="all").dropna(axis=1, how="all")
    common = corr.index.intersection(corr.columns)
    corr = corr.loc[common, common]
    return corr.fillna(0.0)

def compute_linkage(corr, method="ward"):
    dist_matrix = 1 - corr.values
    np.fill_diagonal(dist_matrix, 0)
    dist_matrix = (dist_matrix + dist_matrix.T) / 2
    dist_matrix = np.clip(dist_matrix, 0, None)
    condensed = squareform(dist_matrix, checks=False)
    Z = linkage(condensed, method=method)
    return Z

def plot_corr_clustermap_with_clusters(corr, threshold, method, fig_size, label_fontsize, title):
    Z = compute_linkage(corr, method)
    cluster_labels = fcluster(Z, t=threshold, criterion="distance")
    
    cluster_df = pd.DataFrame({
        "celltype": corr.index,
        "cluster": cluster_labels
    })
    
    n_clusters = cluster_df["cluster"].nunique()
    cluster_palette = dict(zip(
        sorted(cluster_df["cluster"].unique()),
        sns.color_palette("tab20", n_clusters)
    ))
    
    cluster_map = cluster_df.set_index("celltype")["cluster"].to_dict()
    row_colors = pd.Series(
        [cluster_palette[cluster_map[ct]] for ct in corr.index],
        index=corr.index,
        name="Cluster"
    )
    
    cbar_pos = (0.02, 0.8, 0.03, 0.15) if SHOW_CBAR else None
    
    g = sns.clustermap(
        corr,
        cmap=CMAP,
        vmin=VMIN,
        vmax=VMAX,
        linewidths=0,
        figsize=(fig_size, fig_size),
        dendrogram_ratio=0.1,
        cbar_pos=cbar_pos,
        tree_kws={"linewidths": 0.5},
        row_colors=row_colors,
        col_colors=row_colors,
        row_linkage=Z,
        col_linkage=Z,
    )
    
    ax = g.ax_heatmap
    ax.grid(False)
    
    rows = g.data2d.index
    cols = g.data2d.columns
    
    ax.set_yticks(np.arange(len(rows)) + 0.5)
    ax.set_yticklabels(rows, rotation=0, fontsize=label_fontsize)
    
    ax.set_xticks(np.arange(len(cols)) + 0.5)
    ax.set_xticklabels(cols, rotation=90, fontsize=label_fontsize)
    
    ax.set_aspect("equal")
    
    g.fig.suptitle(title, y=0.98, fontsize=TITLE_FONTSIZE)
    
    return g, cluster_df

def build_cluster_dict(cluster_df):
    """Build dict mapping cluster number to list of cell types."""
    cluster_dict = {}
    for c in sorted(cluster_df["cluster"].unique()):
        members = cluster_df[cluster_df["cluster"] == c]["celltype"].tolist()
        cluster_dict[c] = members
    return cluster_dict

def print_clusters(cluster_dict, level_name):
    print(f"\n{level_name} clusters ({len(cluster_dict)} total):")
    for c, members in cluster_dict.items():
        print(f"  Cluster {c} ({len(members)}): {', '.join(members)}")

def get_clusters_at_threshold(corr, threshold, method="ward"):
    Z = compute_linkage(corr, method)
    cluster_labels = fcluster(Z, t=threshold, criterion="distance")
    cluster_df = pd.DataFrame({"celltype": corr.index, "cluster": cluster_labels})
    cluster_dict = {}
    for c in sorted(cluster_df["cluster"].unique()):
        members = cluster_df[cluster_df["cluster"] == c]["celltype"].tolist()
        cluster_dict[c] = members
    return cluster_dict, cluster_df

def compute_score(props, cluster_cols, method):
    if method == "sum":
        return props[cluster_cols].sum(axis=1)
    elif method == "mean":
        return props[cluster_cols].mean(axis=1)
    elif method == "max":
        return props[cluster_cols].max(axis=1)
    elif method == "pca":
        from sklearn.preprocessing import StandardScaler
        from sklearn.decomposition import PCA
        X = props[cluster_cols].values
        if X.shape[1] < 2:
            return props[cluster_cols].iloc[:, 0]
        X_scaled = StandardScaler().fit_transform(X)
        pca = PCA(n_components=1)
        return pca.fit_transform(X_scaled).flatten()
    else:
        raise ValueError(f"Unknown method: {method}")

def stratify(scores, method):
    if method == "median":
        threshold = scores.median()
        groups = np.where(scores >= threshold, "High", "Low")
        return groups, None
    elif method == "tertile":
        t33 = scores.quantile(0.33)
        t67 = scores.quantile(0.67)
        groups = np.where(scores >= t67, "High", np.where(scores <= t33, "Low", "Middle"))
        return groups, "Middle"
    elif method == "quartile_vs_rest":
        q75 = scores.quantile(0.75)
        groups = np.where(scores >= q75, "High", "Low")
        return groups, None
    elif method == "quartile_vs_quartile":
        q25 = scores.quantile(0.25)
        q75 = scores.quantile(0.75)
        groups = np.where(scores >= q75, "High", np.where(scores <= q25, "Low", "Middle"))
        return groups, "Middle"
    else:
        raise ValueError(f"Unknown stratification: {method}")

# =============================================================================
# RMST ANALYSIS with MGMT + AGE adjustment
# =============================================================================

def run_rmst_mgmt_age(props_orig, cluster_cols, clin, scoring_method, stratification,
                      time_col, event_col, tau):
    """
    Run RMST analysis using survRM2 with BOTH MGMT and Age adjustment.
    """
    props = props_orig.copy()
    
    try:
        props["cluster_score"] = compute_score(props, cluster_cols, scoring_method)
    except Exception:
        return None
    
    props = props.reset_index()
    props.columns = ["sample_id_raw"] + list(props.columns[1:])
    props = props[props["sample_id_raw"].astype(str).str.endswith("a")].copy()
    props["sample_id"] = props["sample_id_raw"].astype(str).str[:-1]
    
    merged = props[["sample_id", "cluster_score"]].merge(clin, on="sample_id", how="inner")
    
    surv_df = merged[["sample_id", "cluster_score", time_col, event_col]].copy()
    surv_df["mgmt"] = (merged[MGMT_COL].astype(str) == "Methylated").astype(int).values
    surv_df["age"] = pd.to_numeric(merged[AGE_COL], errors="coerce").values
    
    surv_df = surv_df.rename(columns={time_col: "time", event_col: "event"})
    surv_df["time"] = pd.to_numeric(surv_df["time"], errors="coerce")
    surv_df["event"] = pd.to_numeric(surv_df["event"], errors="coerce")
    
    # Drop rows with missing covariates
    surv_df = surv_df.dropna(subset=["time", "event", "cluster_score", "mgmt", "age"])
    surv_df = surv_df[surv_df["time"] > 0]
    
    if len(surv_df) < 20:
        return None
    
    groups, exclude = stratify(surv_df["cluster_score"], stratification)
    surv_df["group"] = groups
    
    if exclude:
        surv_df = surv_df[surv_df["group"] != exclude].copy()
    
    if len(surv_df) < 20:
        return None
    
    surv_df["arm"] = (surv_df["group"] == "High").astype(int)
    
    n_high = (surv_df["group"] == "High").sum()
    n_low = (surv_df["group"] == "Low").sum()
    
    if n_high < 5 or n_low < 5:
        return None
    
    max_time = surv_df["time"].max()
    if tau > max_time:
        return None
    
    with localconverter(ro.default_converter + pandas2ri.converter):
        r_df = ro.conversion.py2rpy(surv_df)
    
    ro.globalenv["surv_data"] = r_df
    ro.globalenv["tau"] = tau
    
    try:
        # Adjusted analysis with BOTH MGMT and Age
        ro.r("""
        library(survRM2)
        
        # Create covariate matrix (MGMT and Age)
        # Standardize age for numerical stability
        surv_data$age_scaled <- scale(surv_data$age)[,1]
        cov_matrix <- cbind(surv_data$mgmt, surv_data$age_scaled)
        
        # Run adjusted RMST analysis
        rmst_adj <- rmst2(
            time = surv_data$time,
            status = surv_data$event,
            arm = surv_data$arm,
            tau = tau,
            covariates = cov_matrix
        )
        
        # Extract adjusted results
        adj_diff <- rmst_adj$adjusted.result[1, "Est."]
        adj_diff_lower <- rmst_adj$adjusted.result[1, "lower .95"]
        adj_diff_upper <- rmst_adj$adjusted.result[1, "upper .95"]
        adj_p <- rmst_adj$adjusted.result[1, "p"]
        
        # Also get unadjusted for comparison
        rmst_unadj <- rmst2(
            time = surv_data$time,
            status = surv_data$event,
            arm = surv_data$arm,
            tau = tau
        )
        rmst_high <- rmst_unadj$RMST.arm1$rmst["Est."]
        rmst_low <- rmst_unadj$RMST.arm0$rmst["Est."]
        unadj_diff <- rmst_unadj$unadjusted.result[1, "Est."]
        unadj_p <- rmst_unadj$unadjusted.result[1, "p"]
        
        # Get MGMT-only adjusted for comparison
        cov_mgmt_only <- as.matrix(surv_data$mgmt)
        rmst_mgmt <- rmst2(
            time = surv_data$time,
            status = surv_data$event,
            arm = surv_data$arm,
            tau = tau,
            covariates = cov_mgmt_only
        )
        mgmt_adj_p <- rmst_mgmt$adjusted.result[1, "p"]
        """)
        
        rmst_diff = ro.r("adj_diff")[0]
        rmst_diff_lower = ro.r("adj_diff_lower")[0]
        rmst_diff_upper = ro.r("adj_diff_upper")[0]
        rmst_p = ro.r("adj_p")[0]
        rmst_high = ro.r("rmst_high")[0]
        rmst_low = ro.r("rmst_low")[0]
        unadj_diff = ro.r("unadj_diff")[0]
        unadj_p = ro.r("unadj_p")[0]
        mgmt_only_p = ro.r("mgmt_adj_p")[0]
        
    except Exception as e:
        return None
    
    return {
        "n_total": len(surv_df),
        "n_high": n_high,
        "n_low": n_low,
        "tau": tau,
        "rmst_high": rmst_high,
        "rmst_low": rmst_low,
        "rmst_diff": rmst_diff,
        "rmst_diff_lower": rmst_diff_lower,
        "rmst_diff_upper": rmst_diff_upper,
        "rmst_p_mgmt_age": rmst_p,  # Adjusted for BOTH
        "rmst_p_mgmt_only": mgmt_only_p,  # Adjusted for MGMT only
        "rmst_p_unadjusted": unadj_p,  # No adjustment
    }

# =============================================================================
# STRATIFICATION HELPER
# =============================================================================

def apply_stratification(scores, method):
    """
    Returns arm assignment (1=high, 0=low) and mask for rows to keep.
    Middle group is excluded for tertile and quartile_vs_quartile.
    """
    if method == "median":
        threshold = scores.median()
        arm  = (scores >= threshold).astype(int)
        keep = pd.Series(True, index=scores.index)
    elif method == "quartile_vs_rest":
        q75  = scores.quantile(0.75)
        arm  = (scores >= q75).astype(int)
        keep = pd.Series(True, index=scores.index)
    elif method == "tertile":
        t33  = scores.quantile(0.33)
        t67  = scores.quantile(0.67)
        arm  = (scores >= t67).astype(int)
        keep = (scores >= t67) | (scores <= t33)
    elif method == "quartile_vs_quartile":
        q25  = scores.quantile(0.25)
        q75  = scores.quantile(0.75)
        arm  = (scores >= q75).astype(int)
        keep = (scores >= q75) | (scores <= q25)
    else:
        threshold = scores.median()
        arm  = (scores >= threshold).astype(int)
        keep = pd.Series(True, index=scores.index)
    return arm, keep

# =============================================================================
# RMST FUNCTION
# =============================================================================

def run_rmst_module(score_series, clin_df, tau, strat_method):
    score_df = score_series.reset_index()
    score_df.columns = ["sample_id", "score"]

    merged = score_df.merge(clin_df, on="sample_id", how="inner")
    merged["time"]  = pd.to_numeric(merged[TIME_COL],  errors="coerce")
    merged["event"] = pd.to_numeric(merged[EVENT_COL], errors="coerce")
    merged = merged.dropna(subset=[
        "time", "event", "score",
        "mgmt_num", "age_z", "sex_num", "p53_num"
    ])
    merged = merged[merged["time"] > 0].copy()

    if len(merged) < 20 or tau > merged["time"].max():
        return None

    arm, keep = apply_stratification(merged["score"], strat_method)
    merged["arm"] = arm
    merged = merged[keep].copy()

    n_high = int(merged["arm"].sum())
    n_low  = int((merged["arm"] == 0).sum())
    if n_high < 5 or n_low < 5:
        return None

    with localconverter(ro.default_converter + pandas2ri.converter):
        r_df = ro.conversion.py2rpy(merged)
    ro.globalenv["surv_data"] = r_df
    ro.globalenv["tau_val"]   = float(tau)

    try:
        ro.r("""
        # Unadjusted
        rmst_u <- rmst2(surv_data$time, surv_data$event, surv_data$arm, tau=tau_val)
        u_diff <- rmst_u$unadjusted.result[1, "Est."]
        u_lo   <- rmst_u$unadjusted.result[1, "lower .95"]
        u_hi   <- rmst_u$unadjusted.result[1, "upper .95"]
        u_p    <- rmst_u$unadjusted.result[1, "p"]

        # MGMT only
        cov_mgmt <- as.matrix(surv_data$mgmt_num)
        rmst_m <- rmst2(surv_data$time, surv_data$event, surv_data$arm,
                        tau=tau_val, covariates=cov_mgmt)
        m_diff <- rmst_m$adjusted.result[1, "Est."]
        m_lo   <- rmst_m$adjusted.result[1, "lower .95"]
        m_hi   <- rmst_m$adjusted.result[1, "upper .95"]
        m_p    <- rmst_m$adjusted.result[1, "p"]

        # MGMT + Age
        surv_data$age_z_col <- scale(surv_data$age_z)[,1]
        cov_mgmt_age <- cbind(surv_data$mgmt_num, surv_data$age_z_col)
        rmst_ma <- rmst2(surv_data$time, surv_data$event, surv_data$arm,
                         tau=tau_val, covariates=cov_mgmt_age)
        ma_diff <- rmst_ma$adjusted.result[1, "Est."]
        ma_lo   <- rmst_ma$adjusted.result[1, "lower .95"]
        ma_hi   <- rmst_ma$adjusted.result[1, "upper .95"]
        ma_p    <- rmst_ma$adjusted.result[1, "p"]

        # All: MGMT + Age + Sex + p53
        cov_all <- cbind(surv_data$mgmt_num, surv_data$age_z_col,
                         surv_data$sex_num, surv_data$p53_num)
        rmst_a <- rmst2(surv_data$time, surv_data$event, surv_data$arm,
                        tau=tau_val, covariates=cov_all)
        a_diff <- rmst_a$adjusted.result[1, "Est."]
        a_lo   <- rmst_a$adjusted.result[1, "lower .95"]
        a_hi   <- rmst_a$adjusted.result[1, "upper .95"]
        a_p    <- rmst_a$adjusted.result[1, "p"]
        """)

        return {
            "n_high":        n_high,
            "n_low":         n_low,
            "strat":         strat_method,
            "none_diff":     float(ro.r("u_diff")[0]),
            "none_lo":       float(ro.r("u_lo")[0]),
            "none_hi":       float(ro.r("u_hi")[0]),
            "none_p":        float(ro.r("u_p")[0]),
            "mgmt_diff":     float(ro.r("m_diff")[0]),
            "mgmt_lo":       float(ro.r("m_lo")[0]),
            "mgmt_hi":       float(ro.r("m_hi")[0]),
            "mgmt_p":        float(ro.r("m_p")[0]),
            "mgmt_age_diff": float(ro.r("ma_diff")[0]),
            "mgmt_age_lo":   float(ro.r("ma_lo")[0]),
            "mgmt_age_hi":   float(ro.r("ma_hi")[0]),
            "mgmt_age_p":    float(ro.r("ma_p")[0]),
            "all_diff":      float(ro.r("a_diff")[0]),
            "all_lo":        float(ro.r("a_lo")[0]),
            "all_hi":        float(ro.r("a_hi")[0]),
            "all_p":         float(ro.r("a_p")[0]),
        }

    except Exception as e:
        print(f"  survRM2 error: {e}")
        return None

# =============================================================================
# RUN RMST PER MODULE — using per-cluster scoring AND stratification
# =============================================================================

def run_modules(tau, clin_df):
    print(f"\nRunning RMST (tau={tau}y, adjustment={ADJUST_MODE})...")

    cluster_results = {}

    for cid in sig_cluster_ids:
        members = cluster_dict_fp.get(cid, [])
        cols    = [c for c in members if c in props_fp.columns]
        if not cols:
            print(f"  Module {cid}: no matching columns — skipping")
            continue

        row_opt = rmst_mgmt_age_optimal[rmst_mgmt_age_optimal["cluster"] == cid]
        scoring = row_opt["scoring"].iloc[0]       if len(row_opt) else "mean"
        strat   = row_opt["stratification"].iloc[0] if len(row_opt) else "quartile_vs_rest"

        if scoring == "mean":
            score = props_fp[cols].mean(axis=1)
        elif scoring == "sum":
            score = props_fp[cols].sum(axis=1)
        elif scoring == "max":
            score = props_fp[cols].max(axis=1)
        elif scoring == "pca" and len(cols) >= 2:
            X_s   = StandardScaler().fit_transform(props_fp[cols].values)
            score = pd.Series(
                PCA(n_components=1).fit_transform(X_s).flatten(),
                index=props_fp.index)
        else:
            score = props_fp[cols].mean(axis=1)

        res = run_rmst_module(score, clin_df, tau, strat)

        if res:
            cluster_results[cid] = res
            d = res[f"{ADJUST_MODE}_diff"] * 12
            p = res[f"{ADJUST_MODE}_p"]
            print(f"  Module {cid} ({scoring}, {strat}): "
                  f"diff={d:+.2f}mo  p={p:.4f}  "
                  f"n_high={res['n_high']}  n_low={res['n_low']}")
        else:
            print(f"  Module {cid}: RMST failed")

    print(f"\nResults: {len(cluster_results)}/{len(sig_cluster_ids)} modules")
    return cluster_results

def fmt_p(p):
    if np.isnan(p):
        return "—"
    if p < 0.0001:
        return f"{p:.2e}"
    return f"{p:.4f}"

def get_vals(res, mode):
    return (
        res[f"{mode}_diff"] * 12,
        res[f"{mode}_lo"]   * 12,
        res[f"{mode}_hi"]   * 12,
        res[f"{mode}_p"],
    )

def y_frac(y_data):
    return (y_data - y_lo) / (y_hi - y_lo)

## Inputs

Every row of the three processed clinical tables, the obs of the snRNA-seq cohort (no
exclusions) and the curated labels of the EFS annotation key.

In [ ]:
clin_out = AllDataCenters.load_clinical(exclude_idh1_mut=False).reset_index(drop=True)
print("Total rows:", len(clin_out))
print("Totals by center:\n", clin_out.groupby("center").size())

obs_sc = AllDataCenters.load_singlecell(obs_only=True)
labels = annotation.map_levels(obs_sc["celltype_level4_scanvi"], efs_copy=True)
for col in annotation.NEW_COLS:
    obs_sc[col] = labels[col].fillna("Unmapped").to_numpy()
adata_sc = ad.AnnData(obs=obs_sc.reset_index(drop=True))
print(f"snRNA-seq: {adata_sc.n_obs:,} nuclei, {obs_sc[SAMPLE_KEY].nunique()} samples, "
      f"unmapped level-4 labels: {(obs_sc[CT_KEY_L4] == 'Unmapped').sum()}")

## Fig 2d: cell-type co-occurrence modules

Per-sample proportions (doublet labels removed before the proportions), Pearson correlation
across samples, Ward linkage on 1 − r, modules cut at distance 1.0. The level-3 matrix is
computed as in the cell (Methods mention it) but is not a panel.

In [ ]:
# Build proportion matrices
print("Building proportion matrices...")
props_l3 = build_proportion_matrix(adata_sc, CT_KEY_L3, SAMPLE_KEY)
props_l4 = build_proportion_matrix(adata_sc, CT_KEY_L4, SAMPLE_KEY)

print(f"Level 3: {props_l3.shape[0]} samples x {props_l3.shape[1]} cell types")
print(f"Level 4: {props_l4.shape[0]} samples x {props_l4.shape[1]} cell types")

# Compute correlations
print("\nComputing correlations...")
corr_l3 = clean_corr(props_l3.corr(method="pearson"))
corr_l4 = clean_corr(props_l4.corr(method="pearson"))

# Save correlations
output.save_table(OUT, "sc_celltype_level3_correlations", corr_l3)
output.save_table(OUT, "sc_celltype_level4_correlations", corr_l4)

# Plot level 3 with clusters
print("\nPlotting level 3...")
g3, clusters_l3_df = plot_corr_clustermap_with_clusters(
    corr_l3,
    threshold=CLUSTER_THRESHOLD_L3,
    method=CLUSTER_METHOD,
    fig_size=FIG_SIZE_L3, 
    label_fontsize=LABEL_FONTSIZE_L3,
    title=f"Cell type correlations (level 3)"
)
clusters_l3 = build_cluster_dict(clusters_l3_df)
print_clusters(clusters_l3, "Level 3")
output.save(OUT, "celltype_correlation_level3", fig=g3.fig, data=g3.data2d)
plt.show()

# Plot level 4 with clusters
print("\nPlotting level 4...")
g4, clusters_l4_df = plot_corr_clustermap_with_clusters(
    corr_l4,
    threshold=CLUSTER_THRESHOLD_L4,
    method=CLUSTER_METHOD,
    fig_size=FIG_SIZE_L4, 
    label_fontsize=LABEL_FONTSIZE_L4,
    title=f"Cell type correlations (level 4)"
)
clusters_l4 = build_cluster_dict(clusters_l4_df)
print_clusters(clusters_l4, "Level 4")
output.save(OUT, "fig2d_celltype_correlation_level4", fig=g4.fig, data=g4.data2d, panel="Fig 2d")
plt.show()

# Save clusters
output.save_table(OUT, "sc_celltype_level3_clusters", clusters_l3_df, index=False)
output.save_table(OUT, "sc_celltype_level4_clusters", clusters_l4_df, panel="Fig 2d modules", index=False)
print(f"\nSaved to {OUT}")

# Store proportion matrices for survival analysis
props_by_sample_l3 = props_l3
props_by_sample_l4 = props_l4

print("\nOutputs available for survival analysis:")
print("  clusters_l3: dict mapping cluster # -> cell types (level 3)")
print("  clusters_l4: dict mapping cluster # -> cell types (level 4)")
print("  props_by_sample_l3: sample x celltype proportion matrix (level 3)")
print("  props_by_sample_l4: sample x celltype proportion matrix (level 4)")

## Fig 2e, step 1: RMST grid search adjusted for MGMT + age

Module partitions at 7 cut heights × 4 scorings × 4 splits × 4 τ. The configuration with the
most significant modules (threshold 1.0, τ = 1.5 y) and, per module, the best scoring and split
are carried to Fig 2e.

In [ ]:
# =============================================================================
# MAIN ANALYSIS
# =============================================================================

print("Loading R packages...")
ro.r("""
suppressPackageStartupMessages({
    library(survival)
    library(survRM2)
})
""")
print("  survival and survRM2 loaded")

if SURVIVAL_TYPE == "OS":
    time_col, event_col = TIME_COL_OS, EVENT_COL_OS
else:
    time_col, event_col = TIME_COL_PFS, EVENT_COL_PFS

print("Preparing clinical data...")
clin = clin_out.copy()
clin["sample_id"] = clin[SAMPLE_ID_COL].astype(str)
clin["_priority"] = clin[CHRONOLOGY_COL].map({"Baseline": 0, "At recurrence": 1}).fillna(2)
clin = clin.sort_values(["sample_id", "_priority"])
clin = clin.drop_duplicates(subset=["sample_id"], keep="first")
clin = clin.drop(columns=["_priority"])

# =============================================================================
# GRID SEARCH WITH MGMT + AGE ADJUSTMENT
# =============================================================================

print("\n" + "="*80)
print("RMST GRID SEARCH (MGMT + Age adjustment)")
print("="*80)
print(f"Thresholds: {CLUSTER_THRESHOLDS}")
print(f"Tau values: {TAU_OPTIONS}")
print(f"Scoring: {SCORING_METHODS}")
print(f"Stratifications: {STRATIFICATIONS}")
print(f"Covariates: MGMT methylation + Age at diagnosis")
print()

all_results = []
cluster_info = {}

for threshold in CLUSTER_THRESHOLDS:
    print(f"\n--- Threshold {threshold} ---")
    
    cluster_dict, _ = get_clusters_at_threshold(corr_l4, threshold, CLUSTER_METHOD)
    cluster_info[threshold] = cluster_dict
    count = 0
    
    for cluster_num, cluster_celltypes in cluster_dict.items():
        cluster_cols = [c for c in cluster_celltypes if c in props_l4.columns]
        
        if len(cluster_cols) < 2:
            continue
        
        for tau in TAU_OPTIONS:
            for scoring in SCORING_METHODS:
                for strat in STRATIFICATIONS:
                    result = run_rmst_mgmt_age(
                        props_l4, cluster_cols, clin,
                        scoring, strat, time_col, event_col, tau
                    )
                    
                    if result:
                        result["threshold"] = threshold
                        result["cluster"] = cluster_num
                        result["n_celltypes"] = len(cluster_cols)
                        result["celltypes"] = "; ".join(cluster_cols)
                        result["scoring"] = scoring
                        result["stratification"] = strat
                        all_results.append(result)
                        count += 1
    
    print(f"  Valid results: {count}")

print(f"\n\nTotal valid results: {len(all_results)}")

if not all_results:
    raise ValueError("No valid results")

results_df = pd.DataFrame(all_results)
results_df = results_df.sort_values("rmst_p_mgmt_age")

# =============================================================================
# COMPREHENSIVE SUMMARY
# =============================================================================

print("\n" + "="*80)
print("TOP 20 RESULTS (Adjusted for MGMT + Age)")
print("="*80)

display_cols = ["threshold", "cluster", "n_celltypes", "tau", "scoring", "stratification",
                "n_high", "n_low", "rmst_diff", "rmst_p_mgmt_age", "rmst_p_mgmt_only", "rmst_p_unadjusted"]

top20 = results_df.head(20).copy()
top20["rmst_diff"] = top20["rmst_diff"].round(3)
top20["rmst_p_mgmt_age"] = top20["rmst_p_mgmt_age"].round(4)
top20["rmst_p_mgmt_only"] = top20["rmst_p_mgmt_only"].round(4)
top20["rmst_p_unadjusted"] = top20["rmst_p_unadjusted"].round(4)
print(top20[display_cols].to_string(index=False))

# Show cell types for top 10
print("\n--- CELL TYPES IN TOP 10 ---")
for i, (_, row) in enumerate(results_df.head(10).iterrows()):
    diff_mo = row['rmst_diff'] * 12
    direction = "protective" if row['rmst_diff'] > 0 else "harmful"
    print(f"\n{i+1}. Threshold {row['threshold']}, Cluster {row['cluster']} ({direction})")
    print(f"   RMST diff: {diff_mo:+.1f} months, tau={row['tau']}yr")
    print(f"   p(MGMT+Age): {row['rmst_p_mgmt_age']:.4f}, p(MGMT): {row['rmst_p_mgmt_only']:.4f}, p(unadj): {row['rmst_p_unadjusted']:.4f}")
    print(f"   Cells: {row['celltypes']}")

# =============================================================================
# SIGNIFICANT RESULTS SUMMARY
# =============================================================================

print("\n" + "="*80)
print("SIGNIFICANT RESULTS BY THRESHOLD (p < 0.05, MGMT+Age adjusted)")
print("="*80)

sig_all = results_df[results_df["rmst_p_mgmt_age"] < 0.05]

print(f"\nTotal significant results: {len(sig_all)}")

# Count by threshold
print("\n--- By Threshold ---")
thresh_summary = sig_all.groupby("threshold").agg({
    "rmst_p_mgmt_age": ["count", "min"],
    "cluster": "nunique"
}).reset_index()
thresh_summary.columns = ["Threshold", "N_significant", "Best_p", "N_unique_clusters"]
thresh_summary = thresh_summary.sort_values("N_significant", ascending=False)
print(thresh_summary.to_string(index=False))

# Count by tau
print("\n--- By Tau ---")
tau_summary = sig_all.groupby("tau").agg({
    "rmst_p_mgmt_age": ["count", "min"],
    "cluster": "nunique"
}).reset_index()
tau_summary.columns = ["Tau", "N_significant", "Best_p", "N_unique_clusters"]
tau_summary = tau_summary.sort_values("N_significant", ascending=False)
print(tau_summary.to_string(index=False))

# Threshold x Tau
print("\n--- By Threshold × Tau ---")
thresh_tau = sig_all.groupby(["threshold", "tau"]).agg({
    "rmst_p_mgmt_age": ["count", "min"],
    "cluster": "nunique"
}).reset_index()
thresh_tau.columns = ["Threshold", "Tau", "N_significant", "Best_p", "N_unique_clusters"]
thresh_tau = thresh_tau.sort_values("N_unique_clusters", ascending=False)
print(thresh_tau.head(15).to_string(index=False))

# =============================================================================
# OPTIMAL CONFIGURATION
# =============================================================================

print("\n" + "="*80)
print("OPTIMAL CONFIGURATION SELECTION")
print("="*80)

best_configs = []
for thresh in sorted(results_df["threshold"].unique()):
    thresh_data = results_df[results_df["threshold"] == thresh]
    for tau in sorted(thresh_data["tau"].unique()):
        subset = thresh_data[thresh_data["tau"] == tau]
        sig_subset = subset[subset["rmst_p_mgmt_age"] < 0.05]
        
        n_sig = len(sig_subset)
        n_clusters_sig = sig_subset["cluster"].nunique() if n_sig > 0 else 0
        best_p = sig_subset["rmst_p_mgmt_age"].min() if n_sig > 0 else 1.0
        n_protective = len(sig_subset[sig_subset["rmst_diff"] > 0]) if n_sig > 0 else 0
        n_harmful = len(sig_subset[sig_subset["rmst_diff"] < 0]) if n_sig > 0 else 0
        mean_effect = sig_subset["rmst_diff"].abs().mean() * 12 if n_sig > 0 else 0
        
        best_configs.append({
            "threshold": thresh,
            "tau": tau,
            "n_significant": n_sig,
            "n_clusters_sig": n_clusters_sig,
            "best_p": best_p,
            "n_protective": n_protective,
            "n_harmful": n_harmful,
            "mean_effect_months": mean_effect
        })

config_df = pd.DataFrame(best_configs)
config_df = config_df.sort_values(["n_clusters_sig", "n_significant"], ascending=False)

print("\n--- Top Configurations ---")
print(config_df.head(15).to_string(index=False))

best_config = config_df.iloc[0]
print(f"\n*** RECOMMENDED CONFIGURATION ***")
print(f"  Threshold: {best_config['threshold']}")
print(f"  Tau: {best_config['tau']} years")
print(f"  Significant results: {best_config['n_significant']}")
print(f"  Unique significant clusters: {best_config['n_clusters_sig']}")
print(f"  Best p-value: {best_config['best_p']:.4f}")
print(f"  Protective: {best_config['n_protective']}, Harmful: {best_config['n_harmful']}")
print(f"  Mean effect: {best_config['mean_effect_months']:.1f} months")

# =============================================================================
# ALL SIGNIFICANT CLUSTERS AT OPTIMAL CONFIG
# =============================================================================

optimal_thresh = best_config['threshold']
optimal_tau = best_config['tau']

print(f"\n" + "="*80)
print(f"SIGNIFICANT CLUSTERS AT OPTIMAL CONFIG (threshold={optimal_thresh}, tau={optimal_tau})")
print(f"Adjusted for MGMT methylation + Age at diagnosis")
print("="*80)

optimal_results = results_df[
    (results_df["threshold"] == optimal_thresh) &
    (results_df["tau"] == optimal_tau) &
    (results_df["rmst_p_mgmt_age"] < 0.05)
].copy()

optimal_by_cluster = optimal_results.sort_values("rmst_p_mgmt_age").drop_duplicates(subset=["cluster"])
optimal_by_cluster = optimal_by_cluster.sort_values("rmst_p_mgmt_age")

print(f"\n{len(optimal_by_cluster)} unique significant clusters:\n")

for _, row in optimal_by_cluster.iterrows():
    diff_mo = row['rmst_diff'] * 12
    direction = "PROTECTIVE" if row['rmst_diff'] > 0 else "HARMFUL"
    ci_lo = row['rmst_diff_lower'] * 12
    ci_hi = row['rmst_diff_upper'] * 12
    
    print(f"Cluster {row['cluster']} ({row['n_celltypes']} cell types) - {direction}")
    print(f"  p(MGMT+Age) = {row['rmst_p_mgmt_age']:.4f}")
    print(f"  p(MGMT only) = {row['rmst_p_mgmt_only']:.4f}")
    print(f"  p(unadjusted) = {row['rmst_p_unadjusted']:.4f}")
    print(f"  RMST diff: {diff_mo:+.1f} months (95% CI: {ci_lo:+.1f} to {ci_hi:+.1f})")
    print(f"  High: {row['rmst_high']:.2f} yrs, Low: {row['rmst_low']:.2f} yrs")
    print(f"  Best params: {row['scoring']}, {row['stratification']}")
    print(f"  Cell types: {row['celltypes']}")
    print()

# =============================================================================
# PUBLICATION TABLE
# =============================================================================

print("\n" + "="*80)
print("SUMMARY TABLE FOR PUBLICATION")
print("="*80)

pub_table = optimal_by_cluster[[
    "cluster", "n_celltypes", "rmst_diff", "rmst_diff_lower", "rmst_diff_upper",
    "rmst_p_mgmt_age", "rmst_p_mgmt_only", "rmst_p_unadjusted", "celltypes"
]].copy()

pub_table["direction"] = pub_table["rmst_diff"].apply(lambda x: "Protective" if x > 0 else "Harmful")
pub_table["rmst_diff_months"] = pub_table["rmst_diff"] * 12
pub_table["ci_lower_months"] = pub_table["rmst_diff_lower"] * 12
pub_table["ci_upper_months"] = pub_table["rmst_diff_upper"] * 12

pub_table_final = pub_table[[
    "cluster", "n_celltypes", "direction", "rmst_diff_months", 
    "ci_lower_months", "ci_upper_months", 
    "rmst_p_mgmt_age", "rmst_p_mgmt_only", "rmst_p_unadjusted", "celltypes"
]].copy()

pub_table_final.columns = [
    "Cluster", "N_celltypes", "Direction", "RMST_diff_months",
    "CI_lower", "CI_upper", 
    "P_MGMT_Age", "P_MGMT_only", "P_unadjusted", "Cell_types"
]

print(f"\nOptimal configuration: Threshold={optimal_thresh}, Tau={optimal_tau} years")
print(f"Covariates: MGMT methylation + Age at diagnosis\n")
print(pub_table_final.to_string(index=False))

# Save
output.save_table(OUT, "rmst_significant_clusters_mgmt_age_adjusted", pub_table_final, index=False)
output.save_table(OUT, "rmst_grid_search_mgmt_age", results_df, index=False)
print(f"\nSaved to {OUT}/rmst_significant_clusters_mgmt_age_adjusted.csv")

# =============================================================================
# COMPARISON: How many clusters remain significant with full adjustment?
# =============================================================================

print("\n" + "="*80)
print("ROBUSTNESS CHECK: Significance across adjustment levels")
print("="*80)

# For optimal config, compare significance at each level
comparison_data = optimal_results.sort_values("rmst_p_mgmt_age").drop_duplicates(subset=["cluster"])

print(f"\nAt threshold={optimal_thresh}, tau={optimal_tau}:")
print(f"  Significant (p<0.05) unadjusted: {(comparison_data['rmst_p_unadjusted'] < 0.05).sum()}")
print(f"  Significant (p<0.05) MGMT-adjusted: {(comparison_data['rmst_p_mgmt_only'] < 0.05).sum()}")
print(f"  Significant (p<0.05) MGMT+Age-adjusted: {(comparison_data['rmst_p_mgmt_age'] < 0.05).sum()}")

# Store results
rmst_mgmt_age_results = results_df
rmst_mgmt_age_optimal = optimal_by_cluster
optimal_threshold_mgmt_age = optimal_thresh
optimal_tau_mgmt_age = optimal_tau

## Fig 2e: forest of the selected modules adjusted for MGMT + age + sex + p53

In [ ]:
SEL_THRESHOLD = optimal_threshold_mgmt_age

# =============================================================================
# PREPARE CLINICAL DATA
# =============================================================================

clin_fp = clin_out.copy()
clin_fp["sample_id"] = clin_fp[SAMPLE_ID_COL].astype(str)
clin_fp["_priority"] = clin_fp[CHRONOLOGY_COL].map(
    {"Baseline": 0, "At recurrence": 1}).fillna(2)
clin_fp = clin_fp.sort_values(["sample_id", "_priority"])
clin_fp = clin_fp.drop_duplicates(subset=["sample_id"], keep="first")
clin_fp = clin_fp.drop(columns=["_priority"])
clin_fp["mgmt_num"] = (clin_fp[MGMT_COL].astype(str) == "Methylated").astype(float)
clin_fp["sex_num"]  = (clin_fp[SEX_COL].astype(str) == "Female").astype(float)
clin_fp["p53_num"]  = (clin_fp[P53_COL].astype(str) == "Positive").astype(float)
age_vals = pd.to_numeric(clin_fp[AGE_COL], errors="coerce")
clin_fp["age_z"] = (age_vals - age_vals.mean()) / age_vals.std()

print(f"Clinical data: {len(clin_fp)} patients")

# =============================================================================
# BUILD PROPORTION MATRIX AND CLUSTERS
# =============================================================================

print("Building proportion matrix...")
props_fp = props_l4.copy().reset_index()
props_fp.columns = ["sample_id_raw"] + list(props_fp.columns[1:])
props_fp = props_fp[props_fp["sample_id_raw"].astype(str).str.endswith("a")].copy()
props_fp["sample_id"] = props_fp["sample_id_raw"].astype(str).str[:-1]
props_fp = props_fp.drop(columns=["sample_id_raw"]).set_index("sample_id")
print(f"Proportion matrix: {props_fp.shape[0]} samples x {props_fp.shape[1]} cell types")

print("Rebuilding clusters at selected threshold...")
cluster_dict_fp, _ = get_clusters_at_threshold(corr_l4, SEL_THRESHOLD, CLUSTER_METHOD)
sig_cluster_ids = rmst_mgmt_age_optimal["cluster"].tolist()
print(f"Significant modules: {sig_cluster_ids}")

# =============================================================================
# R SETUP
# =============================================================================

ro.r("suppressPackageStartupMessages({library(survival); library(survRM2)})")

cluster_results = run_modules(SEL_TAU, clin_fp)

# =============================================================================
# BUILD ROWS
# =============================================================================

rows = []
for cid, res in cluster_results.items():
    d, lo, hi, p = get_vals(res, ADJUST_MODE)
    sig = p < FP_SIG_ALPHA
    rows.append({
        "label": f"Module {cid}",
        "diff":  d,
        "lo":    lo,
        "hi":    hi,
        "p":     p,
        "sig":   sig,
        "dir":   "pos" if d >= 0 else "neg",
    })

rows_pos = sorted([r for r in rows if r["dir"] == "pos"],
                   key=lambda x: x["diff"], reverse=True)
rows_neg = sorted([r for r in rows if r["dir"] == "neg"],
                   key=lambda x: x["diff"], reverse=True)
fp_rows  = rows_pos + rows_neg
n_rows   = len(fp_rows)
y_pos    = np.arange(n_rows - 1, -1, -1, dtype=float)

# =============================================================================
# PLOT
# =============================================================================

fig, ax = plt.subplots(figsize=(FP_FIG_W, FP_FIG_H))

for i, row in enumerate(fp_rows):
    y   = y_pos[i]
    col = (FP_COL_POS if row["dir"] == "pos" else FP_COL_NEG) if row["sig"] else FP_COL_NS
    ax.errorbar(
        x=row["diff"], y=y,
        xerr=[[row["diff"] - row["lo"]], [row["hi"] - row["diff"]]],
        fmt=FP_MARKER, color=col,
        markersize=FP_MARKER_SZ, capsize=FP_CAP_SZ,
        linewidth=FP_LW, elinewidth=FP_ELW, zorder=3,
    )

ax.axvline(x=0, color="#7F8C8D", linewidth=0.9, linestyle="--", zorder=2)

all_lo = [r["lo"] for r in fp_rows]
all_hi = [r["hi"] for r in fp_rows]
x_pad  = max((max(all_hi) - min(all_lo)) * 0.08, 0.5)
ax.set_xlim(min(all_lo) - x_pad, max(all_hi) + x_pad)
ax.set_ylim(-0.7, n_rows - 0.3)
ax.set_yticks([])
ax.set_xlabel(
    f"RMST difference (months, τ={SEL_TAU}y)  |  high vs low module abundance",
    fontsize=FP_FS_AXIS)
adj_labels = {
    "mgmt":     "MGMT",
    "mgmt_age": "MGMT + Age",
    "all":      "MGMT + Age + Sex + p53",
    "none":     "unadjusted",
}
ax.set_title(
    f"Co-occurrence module survival associations (τ={SEL_TAU} years)\n"
    f"Adjusted for {adj_labels.get(ADJUST_MODE, ADJUST_MODE)}  |  "
    f"blue = protective, red = harmful, gray = n.s.",
    fontsize=FP_FS_TITLE, pad=8)
for sp in ["top", "right", "left"]:
    ax.spines[sp].set_visible(False)
ax.spines["bottom"].set_linewidth(0.6)
ax.tick_params(axis="x", labelsize=FP_FS_AXIS, width=0.6, length=3)
ax.grid(axis="x", linewidth=0.3, alpha=0.4, zorder=0)

trans      = ax.transAxes
y_lo, y_hi = ax.get_ylim()

header_frac = y_frac(n_rows - 0.1)
ax.text(X_LABEL, header_frac, "Module",
        transform=trans, fontsize=FP_FS_HDR, fontweight="bold",
        ha="right", va="bottom")
ax.text(X_EST, header_frac, "RMST diff (months)",
        transform=trans, fontsize=FP_FS_HDR, fontweight="bold",
        ha="left", va="bottom")
ax.text(X_P, header_frac, "adj p",
        transform=trans, fontsize=FP_FS_HDR, fontweight="bold",
        ha="left", va="bottom")

for i, row in enumerate(fp_rows):
    yf  = y_frac(y_pos[i])
    col = (FP_COL_POS if row["dir"] == "pos" else FP_COL_NEG) if row["sig"] else FP_COL_NS
    ax.text(X_LABEL, yf, row["label"],
            transform=trans, fontsize=FP_FS_ROW,
            ha="right", va="center", color="#2C3E50")
    ax.text(X_EST, yf, f"{row['diff']:+.1f}",
            transform=trans, fontsize=FP_FS_ROW,
            ha="left", va="center", color=col)
    ax.text(X_P, yf, fmt_p(row["p"]) + (" *" if row["sig"] else ""),
            transform=trans, fontsize=FP_FS_ROW,
            ha="left", va="center", color=col)

fig.subplots_adjust(left=0.20, right=0.80)
output.save(OUT, "fig2e_forest_rmst_modules", fig=fig, data=pd.DataFrame(fp_rows), panel="Fig 2e")
plt.show()
plt.close(fig)
print(f"Saved {OUT}/fig2e_forest_rmst_modules.png/.pdf")

### Check against the published Fig 2e

In [ ]:
check = pd.DataFrame({cid: {"published diff": d, "this run diff": cluster_results[cid]["all_diff"] * 12,
                            "published p": p, "this run p": cluster_results[cid]["all_p"]}
                      for cid, (d, p) in PUBLISHED_FIG2E.items()}).T
check["match"] = ((check["published diff"] - check["this run diff"]).abs() < 0.005) & \
                 ((check["published p"] - check["this run p"]).abs() < 0.00005)
print(check.round(4).to_string())

### Who is fitted

The rows `run_rmst_module` keeps (same for every module: the score is never missing), and the
fitted patients whose unknown MGMT, sex or p53 the coding counted as the reference level.

In [ ]:
fitted = props_fp[[]].reset_index().merge(clin_fp, on="sample_id", how="inner")
fitted["time"] = pd.to_numeric(fitted[TIME_COL], errors="coerce")
fitted["event"] = pd.to_numeric(fitted[EVENT_COL], errors="coerce")
fitted = fitted.dropna(subset=["time", "event", "mgmt_num", "age_z", "sex_num", "p53_num"])
fitted = fitted[fitted["time"] > 0]
print(f"Fitted patients: {len(fitted)}")
print(f"  not in the 273-patient paper cohort: {int((~fitted['sample_id'].isin(cohort.clinical()['patient'])).sum())}")
print(f"  not in the 260-patient survival cohort: "
      f"{int((~fitted['sample_id'].isin(cohort.survival_cohort()['patient'])).sum())}")
print("Unknown values counted as the reference level (unmethylated / negative / male) among them:")
print(cohort.n_coded_as_reference(fitted).to_string())

### Fig 2e †: modules also significant at τ = 3 years

The Fig 2e models rerun at τ = 3 years (same scoring and splits), and the grid-search rows of
the same modules at τ = 3 years.

In [ ]:
cluster_results_3y = run_modules(DAGGER_TAU, clin_fp)
dagger = [cid for cid, r in cluster_results_3y.items() if r[f"{ADJUST_MODE}_p"] < FP_SIG_ALPHA]
print(f"Significant at tau = {DAGGER_TAU} years (MGMT + age + sex + p53): modules {dagger} (text: 2, 4, 8)")

grid_3y = rmst_mgmt_age_optimal[["cluster", "scoring", "stratification"]].merge(
    rmst_mgmt_age_results[(rmst_mgmt_age_results["threshold"] == optimal_threshold_mgmt_age) &
                          (rmst_mgmt_age_results["tau"] == DAGGER_TAU)],
    on=["cluster", "scoring", "stratification"])
grid_3y["rmst_diff_months"] = grid_3y["rmst_diff"] * 12
print(f"\nGrid search (MGMT + age), tau = {DAGGER_TAU} years:")
print(grid_3y[["cluster", "rmst_diff_months", "rmst_p_mgmt_age", "rmst_p_mgmt_only", "rmst_p_unadjusted"]].round(4).to_string(index=False))
for alpha in (0.05, 0.01):
    print(f"  p(MGMT + age) < {alpha}: modules {grid_3y.loc[grid_3y['rmst_p_mgmt_age'] < alpha, 'cluster'].tolist()}")
output.save_table(OUT, "fig2e_dagger_tau3", pd.DataFrame(
    [{"module": cid, "rmst_diff_months": r["all_diff"] * 12, "p_adjusted": r["all_p"]}
     for cid, r in cluster_results_3y.items()]), panel="Fig 2e dagger", index=False)

### Corrected versions (for the response letter)

(a) Fig 2e on the 260-patient survival cohort with unknown MGMT / sex / p53 kept missing
(complete cases); same modules, scoring and splits. (b) Modules recomputed on the 266 samples
of the paper cohort (loader exclusions), compared with the published 25.

In [ ]:
surv_ids = set(cohort.survival_cohort()["patient"])
clin_fp_corr = clin_fp[clin_fp["sample_id"].isin(surv_ids)].copy()
coded = cohort.code_covariates(clin_fp_corr, missing="keep")
clin_fp_corr["mgmt_num"], clin_fp_corr["sex_num"], clin_fp_corr["p53_num"] = coded["mgmt"], coded["female"], coded["p53_pos"]
print("Corrected (a): survival cohort, complete cases")
cluster_results_corr = run_modules(SEL_TAU, clin_fp_corr)
output.save_table(OUT, "fig2e_corrected_survival_cohort_complete_cases", pd.DataFrame(
    [{"module": cid, "n_high": r["n_high"], "n_low": r["n_low"], "rmst_diff_months": r["all_diff"] * 12,
      "lo": r["all_lo"] * 12, "hi": r["all_hi"] * 12, "p_adjusted": r["all_p"]}
     for cid, r in cluster_results_corr.items()]), index=False)

obs_paper = AllDataCenters.load_singlecell(exclude_idh1_mut=True, exclude_patient_ids=True,
                                         exclude_relapse=True, obs_only=True)
obs_paper[CT_KEY_L4] = annotation.map_levels(obs_paper["celltype_level4_scanvi"], efs_copy=True)[CT_KEY_L4].to_numpy()
props_paper = build_proportion_matrix(ad.AnnData(obs=obs_paper.reset_index(drop=True)), CT_KEY_L4, SAMPLE_KEY)
corr_paper = clean_corr(props_paper.corr(method="pearson"))
modules_paper, _ = get_clusters_at_threshold(corr_paper, CLUSTER_THRESHOLD_L4, CLUSTER_METHOD)
print(f"\nCorrected (b): {props_paper.shape[0]} samples x {props_paper.shape[1]} cell types, "
      f"{len(modules_paper)} modules at threshold {CLUSTER_THRESHOLD_L4}")
as_sets = {frozenset(v) for v in modules_paper.values()}
for cid in sig_cluster_ids:
    print(f"  Module {cid} ({', '.join(clusters_l4[cid])}): "
          f"{'unchanged' if frozenset(clusters_l4[cid]) in as_sets else 'changed'}")
print(f"Modules identical to the published ones: "
      f"{sum(frozenset(v) in as_sets for v in clusters_l4.values())} / {len(clusters_l4)}")

In [ ]:
print(output.versions(("pandas", "numpy", "scipy", "seaborn", "scikit-learn", "rpy2"),
                      ("survival", "survRM2")).to_string(index=False))